# PlotQA : A Retrieval Augmented Question Answering System for Movie Plots

### Setup

Install the necessary libraries for the project

In [1]:
!pip install -U sentence-transformers
!pip install transformers
!pip install faiss-cpu
!pip install -q kaggle
!pip install rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

### Importing Libraries

Import the ncessary libraries

In [2]:
import pandas as pd
import textwrap
import faiss
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import pathlib
import nltk

import warnings
from transformers.utils import logging
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

nltk.download('punkt')
nltk.download('punkt_tab')

from IPython.display import display
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Mounting Google Drive

Mount Google Drive to access the dataset file

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### First time running this notebook (Skip this if already ran this even once)

This section contains a hidden code cell with the commands needed to download the required dataset from Kaggle. This is a one-time setup process. After the dataset has been downloaded and saved to your Google Drive, you do not need to run this section again.

Make a Kaggle API key and save it in your Google Drive.

Make sure to save your Kaggle API key in the Google Drive inside folder "NLP_Project" or change the below path accordingly.

If the file 'wiki_movie_plots_deduped.csv' already exists in the specified path, you can skip running this cell.



In [4]:
KAGGLE_CONFIG_DIR = '/content/drive/MyDrive/NLP_Project'
os.environ["KAGGLE_CONFIG_DIR"] = KAGGLE_CONFIG_DIR

In [5]:
# Confirm the key is present
print("Key found: ", pathlib.Path(KAGGLE_CONFIG_DIR, 'kaggle.json').exists())

Key found:  True


In [6]:
!kaggle datasets download -d jrobischon/wikipedia-movie-plots -p /content
!unzip -o /content/wikipedia-movie-plots.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots
License(s): CC-BY-SA-4.0
 57% 17.0M/29.9M [00:00<00:00, 149MB/s]
100% 29.9M/29.9M [00:00<00:00, 164MB/s]
Archive:  /content/wikipedia-movie-plots.zip
  inflating: /content/data/wiki_movie_plots_deduped.csv  


In [7]:
!cp /content/data/wiki_movie_plots_deduped.csv /content/drive/MyDrive/NLP_Project/

### Reading data

This section loads the main dataset for the project. The
*wiki_movie_plots_deduped.csv* file, which was previously downloaded to Google Drive, is read into a pandas DataFrame for further processing and analysis. The output below shows the dimensions and the first few rows of the dataset to confirm it has been loaded correctly

In [8]:
# Load the dataset from the specified path in Google Drive
df = pd.read_csv("/content/drive/MyDrive/NLP_Project/wiki_movie_plots_deduped.csv")

In [9]:
# Print the shape of the DataFrame to show the number of rows and columns
print(df.shape)
# Display the first 5 rows to get a quick overview of the data
df.head(5)

(34886, 8)


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


### Preprocess and Chunk Plots

This section handles the crucial step of preparing the movie plot data for the RAG system. Since the original plots are often very long, they are broken down into smaller, manageable chunks. This process, known as chunking, ensures that each piece of text is of an optimal size for embedding and retrieval. A word-aware chunking strategy is used to maintain semantic coherence by ensuring sentences are not cut off abruptly. The resulting data is stored in a new DataFrame, chunk_df, ready for the next steps of embedding and indexing.

In [10]:
# Make chunks with some overlap
def chunk_text(text, chunk_size=1024, overlap=200):
    chunks=[]
    tokens = word_tokenize(text)
    num_tokens = len(tokens)
    start=0
    while start < num_tokens:
        end = min(start + chunk_size, num_tokens)
        chunk = " ".join(tokens[start:end])
        chunks.append(chunk)

        if end == num_tokens:
            break

        start += (chunk_size - overlap)
        if start < 0:
            start = 0
    return chunks

In [11]:
# Make chunks and label them with the movie title they are from
chunked_data = []

for idx, row in df.iterrows():
    chunks = chunk_text(row['Plot'])
    for chunk in chunks:
        chunked_data.append((row['Title'], chunk))

In [12]:
# Create a new DataFrame from the chunked data
chunk_df = pd.DataFrame(chunked_data, columns=["Title", "Chunk"])

In [13]:
# Print the new shape of the DataFrame and display the first few rows
print(chunk_df.shape)
chunk_df.head(5)

(36923, 2)


,Title,Chunk
0,Kansas Saloon Smashers,"A bartender is working at a saloon , serving d..."
1,Love by the Light of the Moon,"The moon , painted with a smiling face hangs o..."
2,The Martyred Presidents,"The film , just over a minute long , is compos..."
3,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,Jack and the Beanstalk,The earliest known adaptation of the classic f...


### Encode chunks with Sentence-BERT

This is a critical step in the RAG pipeline where the text chunks are converted into a format that a computer can understand. Using the *SentenceTransformer("all-mpnet-base-v2")* model, each text chunk is encoded into a high-dimensional numerical vector, or embedding.

These embeddings capture the semantic meaning of the text, allowing chunks with similar content to be mathematically closer to each other. The resulting embeddings are stored as a NumPy array, with a shape of (36923, 768)

In [14]:
# Pre-trained Sentence-BERT Model acts as the encoder to conver text to embeddings
model = SentenceTransformer("all-mpnet-base-v2")

# Turn each chunk into a 768-dim vector(embedding) to capture semantic meaning
texts = [t[1] for t in chunked_data]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings)
print("Embeddings shape: ", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1154 [00:00<?, ?it/s]

Embeddings shape:  (36923, 768)


### Build FAISS Index

This step is crucial for enabling fast and efficient retrieval. After creating the sentence embeddings, a FAISS (Facebook AI Similarity Search) index is built. This specialized data structure allows for near-instantaneous searches for the most similar vector embeddings, which is a key component of a RAG system. The embeddings are added to the index, making them ready to be queried.

Create a fast similarity-search structure so nearest chunks can be retrieved in milliseconds.

In [15]:
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)     # IndexFlatL2 uses L2 (Euclidean) distance to find the nearest vectors.
index.add(embeddings)

# Print the total number of vectors in the index to verify all chunks were added
print("Index size: ", index.ntotal)

Index size:  36923


### Retrieve Top-k chunks

This section contains the core retrieval functions of the system, which are designed to fetch the most relevant text chunks from the dataset for a given query. To ensure robust performance, the system uses a hybrid approach that combines both semantic and lexical search methods.

The FAISS-based function performs a similarity search on the encoded embeddings, while the BM25-based function uses keyword matching. The outputs of these functions serve as the candidate pool for the re-ranking step in the RAG pipeline.

**FAISS (Semantic) method**

It encodes the query and returns the top-k chunks with the highest cosine similarity scores.

In [16]:
def retrieve_passages_faiss(query, k=30):
    # Encode user query
    q_vec = model.encode([query])

    # Search FAISS
    distances, indices = index.search(np.array(q_vec), k)

    # Return top-k chunk strings
    return [(chunk_df['Chunk'].iloc[idx], 1 - distances[0][i]) for i, idx in enumerate(indices[0])]

**BM25 (Lexical) method**

BM25 is a lexical search algorithm that finds documents based on keyword frequency and inverse document frequency.

In [17]:
# tokenize chunks
tokenized_corpus = [word_tokenize(chunk.lower()) for chunk in chunk_df['Chunk'].tolist()]
bm25_model = BM25Okapi(tokenized_corpus)

In [18]:
def retrieve_passages_bm25(query, k=30):
    tokenized_query = word_tokenize(query.lower())
    doc_scores = bm25_model.get_scores(tokenized_query)

    # get top k indices by sorting descending
    top_k_indices = np.argsort(doc_scores)[::-1][:k]

    return [(chunk_df['Chunk'].iloc[idx], doc_scores[idx]) for idx in top_k_indices]

In [19]:
# def visualize_similarity_scores(query, top_chunks):
#     query_embedding = model.encode([query])[0]
#     similarities = [util.cos_sim(query_embedding, model.encode([chunk])[0]).item()
#                     for chunk in top_chunks]

#     plt.figure(figsize=(8, 4))
#     plt.bar(range(1, len(similarities)+1), similarities)
#     plt.xticks(range(1, len(similarities)+1))
#     plt.ylim(0, 1)
#     plt.xlabel("Top-k Retrieved Chunks")
#     plt.ylabel("Cosine Similarity")
#     plt.title(f"Similarity Scores for: '{query}'")
#     plt.show()

### Generate Answer using T5

Using a pretrained T5-small model to turn prompts to answers.

T5 acts as the decoder in our RAG system.

We use the t5-small model from Hugging Face's Transformers library as the answer generator in our RAG pipeline.

T5 is a text-to-text model, meaning both input and output are strings. It treats every NLP task as a text generation problem.



In [20]:
qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",    # The `google/flan-t5-base` model is chosen for its strong performance in text-to-text tasks.
    device = 0,
    max_new_tokens=200)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [21]:
def hallucinated_check(answer, context_chunks):
    # Filters out-of-scope words not found in plots
    keywords = ["award", "Oscar", "Golden Globe", "released in", "Box Office", "Rotten To"]
    for word in keywords:
        if word.lower() in answer.lower():
            return True

    # Correctly identify valid feedback message
    if "Sorry, the plot does not contain enough information." in answer.lower():
        return False # Valid fallback

    if not context_chunks:
        return True

    # The semantic consistency check.
    # It ensures the generated answer is semantically similar to the source context provided to T5.
    try:
        answer_embedding = model.encode([answer])[0]

        # Encode each context chunk and find the max similarity
        max_sim_to_context = 0
        for chunk in context_chunks:
            chunk_embedding = model.encode([chunk])[0]
            sim = util.cos_sim(answer_embedding, chunk_embedding).item()
            if sim > max_sim_to_context:
                max_sim_to_context = sim

        # The `answer_context_sim_threshold` is the key tuning parameter for reliability.
        # Strong value of 0.78 prevents most hallucinations
        answer_context_sim_threshold = 0.78

        if max_sim_to_context < answer_context_sim_threshold:
            return True # Answer is too semantically far from the provided context
    except Exception as e:
        print(f"Error during answer-context similarity check: {e}")
        return True

    return False # No obvious hallucination detected


In [22]:
# entire RAG pipeline for a given query.
def answer_question_hybrid_rag(query, k_hybrid=30,
                    min_context_tokens=25,
                    show_graph=False):
    # This threshold is for the final filtering of re-ranked chunks.
    sim_threshold_final_filter = 0.5 if "how many" in query.lower() else 0.0

    # Get candidates from FAISS (Semantic search)
    faiss_candidates = retrieve_passages_faiss(query, k=k_hybrid)

    # Get candidates from BM25 (Lexical search)
    bm25_candidates = retrieve_passages_bm25(query, k=k_hybrid)

    # Combine and deduplicate candidates
    # Unique chunks will be considered, and will be reranked
    combined_chunks = list(set([chunk for chunk, score in faiss_candidates + bm25_candidates]))

    if not combined_chunks:
        print("No initial candidates from FAISS or BM25")
        return "Sorry, the plot does not contain enough information."

    # Rerank combined candidates using Sentence-BERT which uses semantic similarity
    query_embedding = model.encode([query], convert_to_tensor=True)
    chunk_embeddings = model.encode(combined_chunks, convert_to_tensor=True)

    # Calculate cosine similarity between query and all combined chunks
    reranked_scores = util.cos_sim(query_embedding, chunk_embeddings)[0].tolist()

    # Pair chunks with their new reranked scores and sort in descending order
    reranked_chunks_with_scores = sorted(
        zip(combined_chunks, reranked_scores),
        key = lambda x: x[1],
        reverse = True
    )

    # # --- Debugging Output for Hybrid ---
    # print(f"\n--- Debugging: Reranked Chunks for '{query}' ---")
    # # Show the top `k_hybrid` (or fewer if not enough) reranked chunks and their new Sentence-BERT scores
    # for i, (chunk, score) in enumerate(reranked_chunks_with_scores[:min(k_hybrid, len(reranked_chunks_with_scores))]):
    #     print(f"  Reranked Chunk {i+1} (Score: {score:.4f}): {chunk[:100]}...")
    # # --- End Debugging Output ---

    # Select top chunks for T5 based on reranked scores and final threshold
    num_chunks_for_t5 = 3

    # Filter the highest ranked chunks based on final semantic threshold
    selected_chunks_for_t5 = []
    for chunk, score in reranked_chunks_with_scores:
        if score >= sim_threshold_final_filter:
            selected_chunks_for_t5.append(chunk)
            if len(selected_chunks_for_t5) >= num_chunks_for_t5:
                break

        # if current chunk's score is low, stop as it was the highest
        elif len(selected_chunks_for_t5) == 0:
            print(f"  Highest reranked chunk ({reranked_chunks_with_scores[0][1]:.4f}) below final filter threshold ({sim_threshold_final_filter}).")
            return "Sorry, the plot does not contain enough information."

    # If no chunks passed the final filter
    if not selected_chunks_for_t5:
        print("  No chunks passed the final post-reranking filter after all checks.")
        return "Sorry, the plot does not contain enough information."

    context = " ".join(selected_chunks_for_t5)

    # Rule-out if context too small
    if len(context.split()) < min_context_tokens:
        return "Sorry, the plot does not contain enough information."

    # if show_graph:
    #     import matplotlib.pyplot as plt
    #     plt.bar(range(1, k+1), sims); plt.ylim(0,1)
    #     plt.title("Chunk similarities"); plt.show()

    # Build prompt
    max_t5_context_chars = 2000 # can change here with 1000, 1500, 2000
    input_text = f"question: {query} context: {context[:max_t5_context_chars]}"

    # Generate answers with T5
    result = qa_pipeline(input_text)[0]['generated_text']

    # Final fallback in case of weak or hallucinated answers
    if hallucinated_check(result, selected_chunks_for_t5):
        print(f"  Hallucination check triggered for answer: '{result[:50]}...'")
        return "Sorry, the plot does not contain enough information."
    else:
        return result

### Test Set

In [24]:
test_set = [
    # --- Plot 1: Frankenstein (1931 Plot) - 3 Questions ---
    {
        "query": "In Frankenstein, what particular defect does Fritz accidentally introduce into the creature's brain during its acquisition?",
        "ground_truth": "Fritz drops the normal brain he was supposed to steal and has to take the brain of a criminal instead.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Frankenstein, what is the creature's reaction when Maria drowns after he throws her into the lake?",
        "ground_truth": "The creature is puzzled when Maria drowns.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Frankenstein, what sequence of events leads to the creature's final demise at the windmill?",
        "ground_truth": "The creature carries Henry to an old mill, where villagers follow, set the windmill ablaze, and the creature is killed inside.",
        "expected_type": "Correct Answer"
    },

    # --- Plot 2: Oliver Twist - 3 Questions ---
    {
        "query": "In Oliver Twist, what is the unstated, pragmatic reason why the gang of thieves befriends Oliver?",
        "ground_truth": "They befriend him for their own purposes.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Oliver Twist, does the plot summary reveal the exact lineage or identity of Oliver's father?",
        "ground_truth": "No, the plot summary does not provide the exact lineage or identity of Oliver's father; it only states that there are secrets from Oliver's family history waiting to come to light.",
        "expected_type": "Correct Fallback"
    },
    {
        "query": "In Oliver Twist, what two major negative experiences does Oliver face directly after leaving the workhouse?",
        "ground_truth": "He is apprenticed to an uncaring undertaker and subsequently falls into the clutches of a gang of thieves.",
        "expected_type": "Correct Answer"
    },

    # --- Plot 3: Charlie Chaplin (The Love Letter plot) - 3 Questions ---
    {
        "query": "In Charlie Chaplin (The Love Letter plot), what two types of incriminating items are accidentally swapped between Charlie and Ambrose due to their coat mix-up?",
        "ground_truth": "A baby bottle (Ambrose's errand) and a love letter (in Ambrose's coat pocket).",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Charlie Chaplin (The Love Letter plot), how does the coat swap between Charlie and Ambrose occur?",
        "ground_truth": "The coat swap between Charlie and Ambrose occurs accidentally when they meet in a restaurant and mistakenly leave with each other's coats.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Charlie Chaplin (The Love Letter plot), what specific action does Charlie take that reignites the argument between Ambrose and his wife after they had seemingly resolved their initial conflict?",
        "ground_truth": "Charlie sparks another fight by showing Ambrose's wife the love letter that was in Ambrose's pocket.",
        "expected_type": "Correct Answer"
    },

    # --- Plot 4: Cinderella - 3 Questions ---
    {
        "query": "In Cinderella, what three specific items does the fairy godmother transform to create Cinderella's transport to the ball?",
        "ground_truth": "A pumpkin (into a stage coach), mice (into horses), and rats (into servants).",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Cinderella, how does Prince Charming confirm Cinderella's identity after she flees the ball?",
        "ground_truth": "He announces his wish to marry the woman whose foot fits the lost glass slipper, and Cinderella's foot fits it.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Cinderella, does the plot explicitly state the reason why Cinderella intentionally offends the Queen?",
        "ground_truth": "No, the plot states Alice 'unintentionally offends the Queen', but does not provide an explicit reason or mention Cinderella in this context.", # Note: This refers to the Alice plot, but is an adversarial question if the system tries to pull details from a general "offended Queen" context. Needs careful Ground Truth for Cinderella's plot. Let's make this unanswerable from Cinderella plot.
        "expected_type": "Correct Fallback"
    },
    # Revised Cinderella Hard Question (to make it a True Correct Fallback for THIS plot)
    {
        "query": "In Cinderella, does the royal herald offer Cinderella the opportunity to exact revenge on her step-family?",
        "ground_truth": "Yes, the royal heralds give her the opportunity to behead her sisters, but she refuses to.",
        "expected_type": "Correct Answer" # Changed to expect answer now.
    },
    {
        "query": "In Cinderella, how many minutes after midnight was Cinderella allowed to stay at the ball?",
        "ground_truth": "Sorry, the plot only states she must be back before the clock strikes midnight, but does not specify how many minutes she had past midnight.",
        "expected_type": "Correct Fallback"
    },


    # --- Plot 5: The Fall Guy - 3 Questions ---
    {
        "query": "In The Fall Guy, what is Johnny Quinlan's initial problem that makes him desperate?",
        "ground_truth": "Johnny Quinlan loses his job in a drug store and is afraid to tell his wife.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In The Fall Guy, what item does Nifty Herman give Johnny to hold onto, and what does it actually contain?",
        "ground_truth": "Nifty gives Johnny a case of high-priced alcohol to hold onto, but it actually contains a cache of drugs.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In The Fall Guy, what is the professional relationship between Johnny Quinlan and Charles Newton at the end of the film?",
        "ground_truth": "Johnny is rewarded for handling himself well by becoming Charles Newton's assistant.",
        "expected_type": "Correct Answer"
    },


    # --- Plot 6: Possessed - 3 Questions ---
    {
        "query": "In Possessed, what is Marian Martin's social class at the beginning of the film, and what does she aspire to?",
        "ground_truth": "Marian Martin is a factory girl living in the working class section, and she is determined to find a better life.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Possessed, how does Marian initially use Wally Stuart's advice?",
        "ground_truth": "Marian uses Wally's advice on meeting and keeping wealthy men to begin a relationship with his friend Mark Whitney.",
        "expected_type": "Correct Answer"
    },
    {
        "query": "In Possessed, what action does Marian take at Mark Whitney's political rally to protect his reputation?",
        "ground_truth": "Marian steps up from the audience and tells them that she is Mrs. Moreland, and that Mark has always been an honorable man, who once belonged to her, but now belongs to them.",
        "expected_type": "Correct Answer"
    },


    # --- General Unanswerable Question (from before) ---
    {
        "query": "How many Oscars did Frankenstein win?",
        "ground_truth": "Sorry, this information is not available in the provided plot summary.",
        "expected_type": "Correct Fallback"
    }
]

In [25]:
# for q in test_questions:
#     top_chunks_faiss = retrieve_passages_faiss(q)
#     chunk_texts_faiss = [chunk for chunk, score in top_chunks_faiss]
#     visualize_similarity_scores(q, chunk_texts_faiss)

# for q in test_questions:
#     print(f"Question: {q}\nAnswer: {answer_question_hybrid_rag(q)}\n")

### Evaluation

In [35]:
results = []
total_questions = len(test_set)
correct_answers_count = 0
correct_fallbacks_count = 0
incorrect_hallucinations_count = 0
incorrect_fallbacks_count = 0

In [36]:
for i, item in enumerate(test_set):
    query = item["query"]
    ground_truth = item["ground_truth"]
    expected_type = item["expected_type"]

    print(f"\n\n--- Running Query {i+1} ---")
    print(f"Question: {query}")

    # Get a generated answer for the current query
    generated_answer = answer_question_hybrid_rag(query)

    # Check if the generated answer is a fallback message
    is_fallback = "sorry, the plot does not contain enough information." in generated_answer.lower()

    actual_type = ""
    is_correct_overall = False

    # Output was fallback
    if is_fallback:
        # If a fallback was expected, it's a 'Correct Fallback'
        if expected_type == "Correct Fallback":
            actual_type = "Correct Fallback"
            correct_fallbacks_count += 1
            is_correct_overall = True
        # If a fallback was NOT expected, it's an 'Incorrect Fallback'
        else:
            actual_type = "Incorrect Fallback"
            incorrect_fallbacks_count += 1

    # Output gives some answer
    else:
        is_answer_semantically_correct = False
        # If an answer was expected, check its semantic correctness
        if expected_type == "Correct Answer":
            try:
                # Use SBERT to check the semantic similarity between the generated answer and the ground truth
                gen_emb = model.encode([generated_answer])[0]
                gt_emb = model.encode([ground_truth])[0]
                sim = util.cos_sim(gen_emb, gt_emb).item()
                # A similarity score > 0.65 is considered a semantically correct answer
                if sim > 0.65:
                    is_answer_semantically_correct = True
            except Exception as e:
                print(f"Error during semantic check: {e}")
                is_answer_semantically_correct = False

            # Categorize the output based on the semantic check
            if is_answer_semantically_correct:
                actual_type = "Correct Answer"
                correct_answers_count += 1
                is_correct_overall = True
            # If an answer was given but failed the semantic check, it's a hallucination
            else:
                actual_type = "Incorrect Hallucination"
                incorrect_hallucinations_count += 1

        # If an answer was given but a fallback was expected, it's a hallucination
        else:
            actual_type = "Incorrect Hallucination"
            incorrect_hallucinations_count += 1

    # Results of evaluation
    results.append({
        "query": query,
        "ground_truth": ground_truth,
        "generated_answer": generated_answer,
        "expected_type": expected_type,
        "actual_type": actual_type,
        "is_correct_overall": is_correct_overall
    })



--- Running Query 1 ---
Question: In Frankenstein, what particular defect does Fritz accidentally introduce into the creature's brain during its acquisition?
  Hallucination check triggered for answer: 'arm and leg...'


--- Running Query 2 ---
Question: In Frankenstein, what is the creature's reaction when Maria drowns after he throws her into the lake?
  Hallucination check triggered for answer: 'grazed arm , the creature a non-lethal head wound...'


--- Running Query 3 ---
Question: In Frankenstein, what sequence of events leads to the creature's final demise at the windmill?
  Hallucination check triggered for answer: 'Frankenstein's arrival , the creature rips apart t...'


--- Running Query 4 ---
Question: In Oliver Twist, what is the unstated, pragmatic reason why the gang of thieves befriends Oliver?
  Hallucination check triggered for answer: 'for their own purposes...'


--- Running Query 5 ---
Question: In Oliver Twist, does the plot summary reveal the exact lineage or id

### Results

In [37]:
print("\n--- Evaluation Results ---")
for res in results:
    print(f"\nQuestion: {res['query']}")
    print(f"  Ground Truth: {res['ground_truth']}")
    print(f"  Generated: {res['generated_answer']}")
    print(f"  Expected: {res['expected_type']}, Actual: {res['actual_type']}")
    print(f"  Overall Correct: {res['is_correct_overall']}")


--- Evaluation Results ---

Question: In Frankenstein, what particular defect does Fritz accidentally introduce into the creature's brain during its acquisition?
  Ground Truth: Fritz drops the normal brain he was supposed to steal and has to take the brain of a criminal instead.
  Generated: Sorry, the plot does not contain enough information.
  Expected: Correct Answer, Actual: Incorrect Fallback
  Overall Correct: False

Question: In Frankenstein, what is the creature's reaction when Maria drowns after he throws her into the lake?
  Ground Truth: The creature is puzzled when Maria drowns.
  Generated: Sorry, the plot does not contain enough information.
  Expected: Correct Answer, Actual: Incorrect Fallback
  Overall Correct: False

Question: In Frankenstein, what sequence of events leads to the creature's final demise at the windmill?
  Ground Truth: The creature carries Henry to an old mill, where villagers follow, set the windmill ablaze, and the creature is killed inside.
  Gen

In [38]:
print("\n--- Summary Metrics ---")
print(f"Total Questions: {total_questions}")
print(f"Correct Answers (System answered correctly): {correct_answers_count}")
print(f"Correct Fallbacks (System correctly said 'Sorry'): {correct_fallbacks_count}")
print(f"Incorrect Hallucinations (System answered incorrectly): {incorrect_hallucinations_count}")
print(f"Incorrect Fallbacks (System said 'Sorry' but should have answered): {incorrect_fallbacks_count}")


--- Summary Metrics ---
Total Questions: 21
Correct Answers (System answered correctly): 0
Correct Fallbacks (System correctly said 'Sorry'): 4
Incorrect Hallucinations (System answered incorrectly): 1
Incorrect Fallbacks (System said 'Sorry' but should have answered): 16


In [39]:
# Adjusted Accuracy: (Correct Answers + Correct Fallbacks) / Total Questions
overall_success_rate = (correct_answers_count + correct_fallbacks_count) / total_questions
print(f"Overall Success Rate (Accuracy): {overall_success_rate:.2%}")

# Fallback Rate: (Correct Fallbacks + Incorrect Fallbacks) / Total Questions
total_fallbacks = correct_fallbacks_count + incorrect_fallbacks_count
fallback_rate = total_fallbacks / total_questions
print(f"Total Fallback Rate: {fallback_rate:.2%}")

# Hallucination Rate: Incorrect Hallucinations / Total Questions
hallucination_rate = incorrect_hallucinations_count / total_questions
print(f"Observed Hallucination Rate: {hallucination_rate:.2%}")

Overall Success Rate (Accuracy): 19.05%
Total Fallback Rate: 95.24%
Observed Hallucination Rate: 4.76%
